# Medical-VQA — Notebook thực nghiệm CONFIG-DRIVEN

Chỉ cần sửa **cell CONFIG** (epochs, lr, batch, project Comet, repo HF) là mọi run tự áp dụng.
Mỗi run lưu vào HF theo **sub-folder = tên run**. Notebook cũ `run_pipeline.ipynb` vẫn giữ nguyên.

## 0. Setup — lấy code mới nhất + cài thư viện

In [ ]:
# Dán token GitHub vào <PAT> (hoặc copy cell clone từ run_pipeline.ipynb cũ)
GITHUB_CLONE_URL = 'https://<PAT>@github.com/HoangVuSnape/02_Code_DeepLearning.git'

import os
if os.path.isdir('02_Code_DeepLearning/.git'):
    print('Repo da ton tai -> cap nhat code moi nhat (git reset --hard origin/main)...')
    get_ipython().system('cd 02_Code_DeepLearning && git fetch origin && git reset --hard origin/main && git log --oneline -1')
elif not os.path.isdir('.git'):
    get_ipython().system('git clone ' + GITHUB_CLONE_URL)
if os.path.isdir('02_Code_DeepLearning'):
    os.chdir('02_Code_DeepLearning')
print('CWD:', os.getcwd())

In [ ]:
# Cai thu vien. hf_xet + hf-transfer de tai dataset nhanh & tranh loi Xet 403/treo 0%
get_ipython().system('pip install -q datasets tqdm huggingface_hub comet_ml python-dotenv scikit-learn pytest hf-transfer transformers hf_xet')
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('OK — thu vien da cai, hf_transfer bat.')

## 1. CONFIG — CHỈ SỬA Ở ĐÂY

In [ ]:
# ================== CONFIG ==================
EPOCHS    = 20
LR        = 1e-3
PATIENCE  = 5
RL_EPOCHS = 5
RL_LR     = 1e-5

BATCH = {'light': 128, 'heavy': 64, 'gpt2': 32}   # theo do nang model

COMET_PROJECT = 'medvqa-generative-ablation'
HF_REPO       = 'VQA-DeepLearning/vqa-rad-generative'

USE_HF_PUSH = True
USE_COMET   = True
USE_DISCORD = True
print('CONFIG:', EPOCHS, 'epochs | lr', LR, '| repo', HF_REPO, '| comet', COMET_PROJECT)

## 2. RUN MATRIX — mỗi run chỉ khai phần khác nhau

In [ ]:
def _freeze_set(prefix, img, txt, batch, attn):
    base = {'img': img, 'txt': txt, 'dec': 'mlp', 'batch': batch, 'attn': attn}
    return [
        dict(base, name=prefix + '_all', freeze=None),
        dict(base, name=prefix + '_img', freeze='image'),
        dict(base, name=prefix + '_txt', freeze='text'),
        dict(base, name=prefix + '_dec', freeze='decoder'),
    ]

RUNS = {
    'A1': _freeze_set('A1_lstm', 'cnn', 'lstm', 'light', False)
        + _freeze_set('A1_trans', 'cnn', 'transformer', 'light', False),
    'A2': _freeze_set('A2_lstm', 'cnn', 'lstm', 'light', True)
        + _freeze_set('A2_trans', 'cnn', 'transformer', 'light', True),
    'P': [
        {'name': 'P_resnet_lstm',  'img': 'resnet18_frozen', 'txt': 'lstm',        'dec': 'mlp',  'batch': 'heavy', 'attn': True,  'freeze': None},
        {'name': 'P_resnet_trans', 'img': 'resnet18_frozen', 'txt': 'transformer', 'dec': 'mlp',  'batch': 'heavy', 'attn': True,  'freeze': None},
        {'name': 'P_pubmed_mlp',   'img': 'pubmedclip',      'txt': 'pubmedbert',  'dec': 'mlp',  'batch': 'heavy', 'attn': False, 'freeze': None},
        {'name': 'P_pubmed_gpt2',  'img': 'pubmedclip',      'txt': 'pubmedbert',  'dec': 'gpt2', 'batch': 'gpt2',  'attn': False, 'freeze': None},
    ],
}

SMOKE = [
    {'name': 'smoke_test_run', 'img': 'cnn',        'txt': 'lstm',       'dec': 'mlp',  'batch': 'gpt2', 'attn': False, 'freeze': None, 'smoke': True},
    {'name': 'smoke_pubmed',   'img': 'pubmedclip', 'txt': 'pubmedbert', 'dec': 'mlp',  'batch': 'gpt2', 'attn': False, 'freeze': None, 'smoke': True},
    {'name': 'smoke_gpt2',     'img': 'pubmedclip', 'txt': 'pubmedbert', 'dec': 'gpt2', 'batch': 'gpt2', 'attn': False, 'freeze': None, 'smoke': True},
]

# Run dac biet
R_RUN   = {'name': 'R_constrained', 'img': 'pubmedclip', 'txt': 'pubmedbert', 'dec': 'gpt2', 'batch': 'gpt2', 'attn': False, 'freeze': None, 'ckpt': 'P_pubmed_gpt2', 'constrained': True}
D_TRAIN = {'name': 'D1_scst', 'img': 'pubmedclip', 'txt': 'pubmedbert', 'dec': 'gpt2', 'batch': 'gpt2', 'attn': False, 'freeze': None, 'rl': True, 'load_checkpoint': 'runs/P_pubmed_gpt2_best.pt'}
D_EVAL  = {'name': 'D1_scst', 'img': 'pubmedclip', 'txt': 'pubmedbert', 'dec': 'gpt2', 'batch': 'gpt2', 'attn': False, 'freeze': None, 'constrained': True}
GEMMA   = [
    {'name': 'gemma-2b-lora', 'csv': 'gemma_preds/gemma2b_test.csv'},
    {'name': 'gemma-4b-lora', 'csv': 'gemma_preds/gemma4b_test.csv'},
]

print('Tong training run:', sum(len(v) for v in RUNS.values()), '| nhom:', list(RUNS))

## 3. HELPER — dựng lệnh từ CONFIG + chạy qua subprocess

In [ ]:
import subprocess

def _common(push=True):
    f = ['--project_name', COMET_PROJECT]
    if USE_COMET:   f += ['--use_comet']
    if USE_DISCORD: f += ['--use_discord']
    if USE_HF_PUSH and push: f += ['--use_hf_push', '--hf_repo_id', HF_REPO]
    return f

def _model(r):
    f = ['--image_encoder', r['img'], '--text_encoder', r['txt'], '--decoder', r['dec']]
    if r.get('attn'): f += ['--image_attention', '--text_attention']
    fr = r.get('freeze')
    if fr == 'image':   f += ['--freeze_image']
    if fr == 'text':    f += ['--freeze_text']
    if fr == 'decoder': f += ['--freeze_decoder']
    return f

def run_train(r):
    cmd = ['python', 'train.py', '--run_name', r['name']] + _model(r)
    cmd += ['--batch_size', str(BATCH[r['batch']])]
    if r.get('smoke'):
        cmd += ['--smoke']
    else:
        cmd += ['--epochs', str(EPOCHS), '--lr', str(LR), '--patience', str(PATIENCE)]
    if r.get('rl'):
        cmd += ['--rl', '--rl_epochs', str(RL_EPOCHS), '--rl_lr', str(RL_LR), '--load_checkpoint', r['load_checkpoint']]
    cmd += _common(push=not r.get('smoke'))
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=False)

def run_eval(r, push=True):
    name = r['name']
    if r.get('csv'):
        cmd = ['python', 'eval.py', '--predictions_csv', r['csv'], '--save_metrics', 'runs/' + name + '_metrics.json', '--hf_subfolder', name]
    else:
        ckpt = r.get('ckpt', name)
        cmd = ['python', 'eval.py', '--checkpoint', 'runs/' + ckpt + '_best.pt'] + _model(r)
        cmd += ['--save_metrics', 'runs/' + name + '_metrics.json', '--hf_subfolder', name]
        if r.get('constrained'): cmd += ['--constrained_closed']
    if not push: cmd += ['--no_hf_push']
    print('>>', ' '.join(cmd))
    subprocess.run(cmd, check=False)

def run_group(g):
    for r in RUNS[g]: run_train(r)
    for r in RUNS[g]: run_eval(r)
    print('== Xong nhom', g, '==')

## 4. SMOKE TEST (nên chạy trước)

In [ ]:
for r in SMOKE:
    run_train(r)
run_eval(SMOKE[1], push=False)   # eval thu smoke_pubmed (khong push HF)

## 5. Nhóm A1 (không attention)

In [ ]:
run_group('A1')

## 6. Nhóm A2 (có attention)

In [ ]:
run_group('A2')

## 7. Nhóm P (pretrained + generative)

In [ ]:
run_group('P')

## 8. Nhóm R — constrained decoding (eval-only trên P_pubmed_gpt2)

In [ ]:
run_eval(R_RUN)

## 9. Nhóm D — RL / SCST (fine-tune P_pubmed_gpt2)

In [ ]:
run_train(D_TRAIN)
run_eval(D_EVAL)

## 10. Nhóm G — Gemma benchmarks (eval CSV)

In [ ]:
for r in GEMMA:
    run_eval(r)

## 11. Backup gộp (an toàn) + Bảng tổng hợp

In [ ]:
# Backup du phong: day toan bo runs/ trong 1 commit (phong khi push tung run bi 429)
from huggingface_hub import HfApi
import os
try:
    HfApi(token=os.environ['HF_TOKEN']).upload_folder(folder_path='runs', repo_id=HF_REPO, repo_type='model', allow_patterns=['*.json', '*.csv', '*.pt'])
    print('Backup toan bo runs/ trong 1 commit')
except Exception as e:
    print('Backup bo qua:', e)

In [ ]:
get_ipython().system('python report.py')